In [1]:
import pip


%pip install opencv-python mediapipe==0.10.14


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
# Capture a hand-gesture YOLO test set
# Install dependencies in a notebook cell if needed:

# Run the cells in order. Show each gesture when prompted; press q to stop.

# Imports and output folders

from pathlib import Path
import re
import time

import cv2
import mediapipe as mp
from mediapipe.python.solutions.hands import Hands

# Gesture -> class ID used by the trained model (same order as names in data.yaml).
GESTURE_CLASS_IDS = {
    "Sorry": 12,
    "Help": 6,
    "Get": 3,
    "Good": 4,
    "I": 0,
    "Love": 9,
    "You": 16,
    "Want": 14,
}
GESTURES = list(GESTURE_CLASS_IDS)
SAMPLES_PER_GESTURE = 20
INTERVAL_SECONDS = 3
CAMERA_INDEX = 0

DATASET_DIR = Path("test_selfie")
IMAGE_DIR = DATASET_DIR / "images"
LABEL_DIR = DATASET_DIR / "labels"
IMAGE_DIR.mkdir(parents=True, exist_ok=True)
LABEL_DIR.mkdir(parents=True, exist_ok=True)


def slug(text):
    return re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")

# Each gesture's class ID comes from GESTURE_CLASS_IDS. Labels use YOLO format:
# class_id x_center y_center width height, with normalized coordinates.

# Webcam capture and automatic hand bounding-box labels
camera = None
stopped = False
try:
    with Hands(
        static_image_mode=False,
        max_num_hands=1,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    ) as detector:
        for gesture, class_id in GESTURE_CLASS_IDS.items():
            camera = cv2.VideoCapture(CAMERA_INDEX)
            if not camera.isOpened():
                raise RuntimeError("Could not open webcam; check CAMERA_INDEX and permissions.")
            count = 0
            prefix = slug(gesture)
            print(f"Show the '{gesture}' gesture. Collecting {SAMPLES_PER_GESTURE} images.")

            x1 = y1 = x2 = y2 = 0
            while count < SAMPLES_PER_GESTURE:
                ok, frame = camera.read()
                if not ok:
                    raise RuntimeError("Failed to read a frame from the webcam.")

                height, width = frame.shape[:2]
                result = detector.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                preview = frame.copy()
                cv2.putText(
                    preview,
                    f"{gesture}  Shots: {count}/{SAMPLES_PER_GESTURE}  (q=quit)",
                    (12, 30),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (0, 255, 0),
                    2,
                    cv2.LINE_AA,
                )

                if result.multi_hand_landmarks:
                    points = result.multi_hand_landmarks[0].landmark
                    x1 = max(0, min(width - 1, int(min(p.x for p in points) * width)))
                    y1 = max(0, min(height - 1, int(min(p.y for p in points) * height)))
                    x2 = max(0, min(width - 1, int(max(p.x for p in points) * width)))
                    y2 = max(0, min(height - 1, int(max(p.y for p in points) * height)))
                    cv2.rectangle(preview, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    cv2.putText(
                        preview,
                        gesture,
                        (x1, max(22, y1 - 8)),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.7,
                        (0, 255, 0),
                        2,
                    )

                cv2.imshow("Gesture test-set capture", preview)
                if cv2.waitKey(1) & 0xFF == ord("q"):
                    stopped = True
                    break

                if result.multi_hand_landmarks and x2 > x1 and y2 > y1:
                    stem = f"{prefix}_{count + 1:02d}"
                    image_path = IMAGE_DIR / f"{stem}.jpg"
                    label_path = LABEL_DIR / f"{stem}.txt"
                    x_center = ((x1 + x2) / 2) / width
                    y_center = ((y1 + y2) / 2) / height
                    box_width = (x2 - x1) / width
                    box_height = (y2 - y1) / height

                    if not cv2.imwrite(str(image_path), frame):
                        raise RuntimeError(f"Could not save image: {image_path}")

                    label_path.write_text(
                        f"{class_id} {x_center:.6f} {y_center:.6f} {box_width:.6f} {box_height:.6f}\n",
                        encoding="utf-8",
                    )
                    count += 1
                    print(f"Saved {stem} ({count}/{SAMPLES_PER_GESTURE})")

                    print(f"Prepare to show the '{gesture}' gesture before the next capture countdown.")
                    deadline = time.monotonic() + INTERVAL_SECONDS
                    while time.monotonic() < deadline:
                        ok, wait_frame = camera.read()
                        if ok:
                            remaining = max(0, int(deadline - time.monotonic()) + 1)
                            cv2.putText(
                                wait_frame,
                                f"Show: {gesture}",
                                (12, 30),
                                cv2.FONT_HERSHEY_SIMPLEX,
                                0.8,
                                (0, 255, 0),
                                2,
                                cv2.LINE_AA,
                            )
                            cv2.putText(
                                wait_frame,
                                f"Shots taken: {count}/{SAMPLES_PER_GESTURE} | Next in {remaining}s (q=quit)",
                                (12, 70),
                                cv2.FONT_HERSHEY_SIMPLEX,
                                0.7,
                                (0, 255, 0),
                                2,
                                cv2.LINE_AA,
                            )
                            cv2.imshow("Gesture test-set capture", wait_frame)
                        if cv2.waitKey(1) & 0xFF == ord("q"):
                            stopped = True
                            break
                    if stopped:
                        break

            # Close the webcam window after each word; it reopens for the next one.
            camera.release()
            cv2.destroyAllWindows()
            cv2.waitKey(1)
            if stopped:
                break
            print(f"Finished '{gesture}': {count}/{SAMPLES_PER_GESTURE} shots taken.")
            time.sleep(2)
finally:
    if camera is not None:
        camera.release()
    cv2.destroyAllWindows()

print("Capture stopped early." if stopped else f"Dataset saved to {DATASET_DIR.resolve()}")


Show the 'Sorry' gesture. Collecting 20 images.
Saved sorry_01 (1/20)
Prepare to show the 'Sorry' gesture before the next capture countdown.
Saved sorry_02 (2/20)
Prepare to show the 'Sorry' gesture before the next capture countdown.
Saved sorry_03 (3/20)
Prepare to show the 'Sorry' gesture before the next capture countdown.
Saved sorry_04 (4/20)
Prepare to show the 'Sorry' gesture before the next capture countdown.
Saved sorry_05 (5/20)
Prepare to show the 'Sorry' gesture before the next capture countdown.
Saved sorry_06 (6/20)
Prepare to show the 'Sorry' gesture before the next capture countdown.
Saved sorry_07 (7/20)
Prepare to show the 'Sorry' gesture before the next capture countdown.
Saved sorry_08 (8/20)
Prepare to show the 'Sorry' gesture before the next capture countdown.
Saved sorry_09 (9/20)
Prepare to show the 'Sorry' gesture before the next capture countdown.
Saved sorry_10 (10/20)
Prepare to show the 'Sorry' gesture before the next capture countdown.
Saved sorry_11 (11/20